## Incorporating Arrival & Departure Timing Features

Goal: Building from baseline classification model that predicts whether a sequences's final actual arrival time differs from its estimated arrival time.

In [14]:
## Load Libraties and Data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

Importing libraries needed for data manipulation, visualization,splitting the data, and building a baseline classification model.

In [ ]:
#Loading Data & Minor Cleansing
seq_spoil = pd.read_csv("../data/raw/sequence_spoilage.csv", sep="|")
assign_history = pd.read_csv("../data/raw/Assignment_History.csv")

seq_spoil = seq_spoil.loc[:, ~seq_spoil.columns.str.contains("^Unnamed")]
assign_history = assign_history.loc[:, ~assign_history.columns.str.contains("^Unnamed")]

seq_spoil.columns = seq_spoil.columns.str.upper().str.strip()
assign_history.columns = assign_history.columns.str.upper().str.strip()

print(seq_spoil.columns)
##print(assign_history.columns)

##print(seq_spoil['FLEET'].unique)
# diff fleets: 737, 320, 777, 787
# typically diff fleets have operational characteristics
spoil = seq_spoil
spoil['FLEET'] = spoil['FLEET'].astype(str)
spoil["FLEET"].value_counts()

#DFW, MIA, CLT, ORD, LAX, PHL, LGA, DCA, PHX, BOS
spoil["BASE"].value_counts()

# DIVISION: International or Domestic
spoil["DIVISION"].value_counts()

#SPOILAGE: Not Spoiled, Partially Spoiled, Fully Spoiled
spoil["SPOILAGE"].value_counts()

# Most Common Flight Patterns: 175-176, 281-281, 61-60, 50-51, 240-239
spoil["FLIGHT_PATTERN"].value_counts()

#Most Common Seq Patterns: DFW-LHR-DFW, DFW-HND-DFW, DFW-ICN-DFW, DFW-CUN-DFW, DFW-FCO-DFW 
spoil["SEQ_PATTERN"].value_counts()
#idea: to group by these variables and look at time

# 2 is minimum and most common, 3,4,5,6,7,8,9,10,11 is max and least common
spoil["SEQ_TTL_LEGS"].value_counts()


Index(['SEQ_NBR', 'SEQ_SCHD_START_DT', 'FLEET', 'BASE', 'DIVISION', 'SPOILAGE',
       'TOTAL_BLOCKED_HRS', 'TOTAL_SPOILED_HRS', 'SEQ_CAL_DAYS',
       'SEQ_DUTY_DAYS', 'SEQ_TTL_FLTTIME', 'MIN_FLYTIME_PER_LEG',
       'MAX_LEGS_PER_DAY', 'SEQ_TTL_LEGS', 'MORETHAN2_321_LEGS', 'IN_SEQ_DHD',
       'LAYOVER', 'SEQ_PATTERN', 'SEQ_START', 'FLIGHT_PATTERN',
       'SEQ_START_HRS', 'SF_LOAD_TMS'],
      dtype='str')


/var/folders/0z/sjg2lccn1mg5jm9bz_pldnbc0000gn/T/ipykernel_74315/3613316908.py:3: DtypeWarning: Columns (0: SEQ_DUTY_PERIOD_CT, 1: CKP_ASSIGNED_IND) have mixed types. Specify dtype option on import or set low_memory=False.
  assign_history = pd.read_csv("../data/raw/Assignment_History.csv")


SEQ_TTL_LEGS
2     4107
6     1583
4     1455
5      719
8      619
7      397
3       83
9       43
10      18
11       1
Name: count, dtype: int64

Loaded both datasets and removed unnamed columns that came from the file export. I also standardized the column names by making them uppercase and stripping extra spaces so I could work with them more consistently later.